# 🕵️ Dia 11 — Triagem de logs web com grep, awk e sed

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 11* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-11.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

O Juliano confirmou: copiou os logs com `cp -p` e mandou os hashes de origem. Agora você abre o `access.log` do portal **real**, `serraazul.example`. Rafa: *"Kit preguiçoso carrega o logo do site original. Cada vítima pede a imagem pra Serra Azul e diz de onde veio, no Referer."*

Neste notebook você vai:

1. **Escrever um parser** do formato combined com uma expressão regular explicada pedaço a pedaço.
2. **Refazer os one-liners do original** em Python, ao lado do `awk` equivalente.
3. **Achar no log da Serra Azul** quem copiou o site e quantos navegadores abriram a página falsa.
4. **Jogar o quiz "qual comando responde?"** e gerar a planilha de triagem com histograma.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo funciona **sem internet**; quando uma célula tenta a rede, ela tem plano B embutido.

## 1. Um parser do formato combined

O `awk` separa por espaço ou por aspas. Em Python, uma expressão regular com **grupos nomeados** faz as duas separações de uma vez e devolve um dicionário por linha.

In [ ]:
import re
from datetime import datetime, timezone

# Cada (?P<nome>...) captura um campo e dá nome a ele.
COMBINED = re.compile(
    r'(?P<ip>\S+) '            # $1   IP do cliente (\S+ = tudo até o próximo espaço)
    r'\S+ \S+ '                # ident e usuário (quase sempre "- -")
    r'\[(?P<hora>[^\]]+)\] '   # $4   [10/Mar/2026:14:02:11 +0000] -> só o miolo
    r'"(?P<metodo>\S+) '       # $6   GET/POST, sem a aspa
    r'(?P<caminho>\S+) '       # $7   /login
    r'[^"]*" '                 #      o protocolo (HTTP/1.1) e a aspa que fecha
    r'(?P<status>\d{3}) '      # $9   200, 404...
    r'(?P<bytes>\S+) '         # $10  tamanho (pode ser "-")
    r'"(?P<referer>[^"]*)" '   # -F'"' $4  Referer
    r'"(?P<ua>[^"]*)"'         # -F'"' $6  User-Agent
)

def ler_log(texto):
    """Transforma o texto do log numa lista de dicionários, com a hora já em UTC."""
    eventos = []
    for n, linha in enumerate(texto.splitlines(), start=1):
        m = COMBINED.match(linha)
        if not m:                                   # linha fora do formato: avisa e segue
            print(f"⚠️ linha {n} fora do formato combined")
            continue
        e = m.groupdict()
        local = datetime.strptime(e["hora"], "%d/%b/%Y:%H:%M:%S %z")   # %z entende +0000 e -0300
        e["utc"] = local.astimezone(timezone.utc)   # converte para UTC: a régua comum de qualquer caso
        e["linha"] = n                               # guarda o número da linha: rastreabilidade
        eventos.append(e)
    return eventos

print("Parser pronto.")

## 2. Os one-liners do original, em Python

Primeiro com o log da loja fictícia `shop.example.org` (o mesmo do README). Cada bloco mostra o comando `awk` e a versão Python. O `Counter` faz o papel de `sort | uniq -c | sort -rn`.

In [ ]:
from collections import Counter

LOG_LOJA = '198.51.100.23 - - [10/Mar/2026:14:02:11 +0000] "GET / HTTP/1.1" 200 5120 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"\n198.51.100.23 - - [10/Mar/2026:14:02:12 +0000] "GET /static/app.css HTTP/1.1" 200 1834 "https://shop.example.org/" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"\n192.0.2.77 - - [10/Mar/2026:14:05:40 +0000] "GET /login HTTP/1.1" 200 2210 "-" "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_4)"\n192.0.2.77 - - [10/Mar/2026:14:05:52 +0000] "POST /login HTTP/1.1" 302 0 "https://shop.example.org/login" "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_4)"\n203.0.113.45 - - [10/Mar/2026:15:10:03 +0000] "GET /wp-login.php HTTP/1.1" 404 512 "-" "python-requests/2.31.0"\n203.0.113.45 - - [10/Mar/2026:15:10:03 +0000] "GET /.env HTTP/1.1" 404 512 "-" "python-requests/2.31.0"\n203.0.113.45 - - [10/Mar/2026:15:10:04 +0000] "GET /admin/config.php HTTP/1.1" 404 512 "-" "python-requests/2.31.0"\n203.0.113.45 - - [10/Mar/2026:15:10:04 +0000] "GET /index.php?page=../../../../etc/passwd HTTP/1.1" 400 312 "-" "python-requests/2.31.0"\n203.0.113.45 - - [10/Mar/2026:15:10:05 +0000] "GET /index.php?page=%2e%2e%2f%2e%2e%2fetc%2fpasswd HTTP/1.1" 400 312 "-" "python-requests/2.31.0"\n203.0.113.45 - - [10/Mar/2026:15:10:05 +0000] "GET /.git/config HTTP/1.1" 403 290 "-" "python-requests/2.31.0"\n198.51.100.140 - - [10/Mar/2026:15:31:18 +0000] "POST /login HTTP/1.1" 200 2290 "-" "Mozilla/5.0 (X11; Linux x86_64)"\n198.51.100.140 - - [10/Mar/2026:15:31:19 +0000] "POST /login HTTP/1.1" 200 2290 "-" "Mozilla/5.0 (X11; Linux x86_64)"\n198.51.100.140 - - [10/Mar/2026:15:31:19 +0000] "POST /login HTTP/1.1" 200 2290 "-" "Mozilla/5.0 (X11; Linux x86_64)"\n198.51.100.140 - - [10/Mar/2026:15:31:20 +0000] "POST /login HTTP/1.1" 200 2290 "-" "Mozilla/5.0 (X11; Linux x86_64)"\n198.51.100.140 - - [10/Mar/2026:15:31:20 +0000] "POST /login HTTP/1.1" 302 0 "-" "Mozilla/5.0 (X11; Linux x86_64)"\n198.51.100.140 - - [10/Mar/2026:15:31:22 +0000] "GET /account/orders HTTP/1.1" 200 8841 "-" "Mozilla/5.0 (X11; Linux x86_64)"\n198.51.100.23 - - [10/Mar/2026:16:44:09 +0000] "GET /products/42 HTTP/1.1" 200 6021 "https://shop.example.org/" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"\n192.0.2.77 - - [10/Mar/2026:16:50:31 +0000] "GET /account/orders HTTP/1.1" 200 8120 "https://shop.example.org/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_4)"\n203.0.113.45 - - [10/Mar/2026:17:02:44 +0000] "GET /server-status HTTP/1.1" 403 290 "-" "python-requests/2.31.0"\n198.51.100.23 - - [10/Mar/2026:17:15:00 +0000] "GET /cart HTTP/1.1" 500 1024 "https://shop.example.org/products/42" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"\n'
loja = ler_log(LOG_LOJA)
print(len(loja), "requisições lidas\n")

def ranking(titulo, comando, valores):
    """Imprime como o 'uniq -c | sort -rn' imprimiria."""
    print(f"# {titulo}\n$ {comando}")
    for valor, n in Counter(valores).most_common():   # most_common = sort -rn
        print(f"{n:>4} {valor}")
    print()

ranking("Quem está mais ativo?", "awk '{print $1}' access.log | sort | uniq -c | sort -rn", [e["ip"] for e in loja])
ranking("Que status voltaram?", "awk '{print $9}' access.log | sort | uniq -c | sort -rn", [e["status"] for e in loja])
ranking("Que clientes são declarados?", "awk -F'\"' '{print $6}' access.log | sort | uniq -c | sort -rn", [e["ua"] for e in loja])

print("# Erros 4xx com hora, origem e caminho")
for e in loja:
    if e["status"].startswith("4"):
        print(e["hora"][:20], e["ip"], e["status"], e["caminho"])

print("\n# Path traversal (puro ou codificado)")
for e in loja:
    if re.search(r"\.\./|%2e%2e", e["caminho"], re.I):   # re.I = o -i do grep
        print(e["ip"], e["caminho"])

In [ ]:
# ------------------------------------------------------------
# A pergunta do login: falhas (200) seguidas de sucesso (302) do mesmo IP
# ------------------------------------------------------------
falhas = Counter()
for e in loja:
    if e["metodo"] == "POST" and e["caminho"] == "/login":
        if e["status"] == "200":                    # nesta aplicação, 200 = formulário de novo = falhou
            falhas[e["ip"]] += 1
        elif e["status"] == "302":                  # 302 = redirecionou = entrou
            print(f"{e['utc']:%H:%M:%S}Z  {e['ip']} ENTROU depois de {falhas[e['ip']]} falha(s)")

## 3. ➕ O portal da Serra Azul

Agora o log real do caso: [`caso-continuo/dados/dia-11/access-serraazul.log`](https://github.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/blob/main/caso-continuo/dados/dia-11/access-serraazul.log). Fuso `-0300`, sem proxy na frente.

In [ ]:
# ------------------------------------------------------------
# Carrega o arquivo do caso: primeiro tenta o repositório no GitHub,
# depois cai na cópia embutida (offline-first: o notebook nunca trava).
# ------------------------------------------------------------
import urllib.request                      # cliente HTTP da biblioteca padrão

URL = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-11/access-serraazul.log"
EMBUTIDO = '198.51.100.31 - - [03/Oct/2026:10:14:02 -0300] "GET / HTTP/1.1" 200 18422 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"\n198.51.100.31 - - [03/Oct/2026:10:14:03 -0300] "GET /static/css/portal.css HTTP/1.1" 200 4410 "https://serraazul.example/" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"\n203.0.113.77 - - [03/Oct/2026:22:41:09 -0300] "GET / HTTP/1.1" 200 18422 "-" "Wget/1.21.4"\n203.0.113.77 - - [03/Oct/2026:22:41:09 -0300] "GET /robots.txt HTTP/1.1" 200 68 "-" "Wget/1.21.4"\n203.0.113.77 - - [03/Oct/2026:22:41:10 -0300] "GET /static/css/portal.css HTTP/1.1" 200 4410 "https://serraazul.example/" "Wget/1.21.4"\n203.0.113.77 - - [03/Oct/2026:22:41:10 -0300] "GET /static/js/portal.js HTTP/1.1" 200 9120 "https://serraazul.example/" "Wget/1.21.4"\n203.0.113.77 - - [03/Oct/2026:22:41:10 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul.example/" "Wget/1.21.4"\n203.0.113.77 - - [03/Oct/2026:22:41:11 -0300] "GET /acesso HTTP/1.1" 200 7311 "https://serraazul.example/" "Wget/1.21.4"\n203.0.113.77 - - [03/Oct/2026:22:41:11 -0300] "GET /static/img/cadeado.svg HTTP/1.1" 200 1022 "https://serraazul.example/acesso" "Wget/1.21.4"\n203.0.113.9 - - [04/Oct/2026:15:20:41 -0300] "GET /wp-login.php HTTP/1.1" 404 512 "-" "python-requests/2.31.0"\n203.0.113.9 - - [04/Oct/2026:15:20:41 -0300] "GET /.env HTTP/1.1" 404 512 "-" "python-requests/2.31.0"\n203.0.113.9 - - [04/Oct/2026:15:20:42 -0300] "GET /index.php?page=../../../../etc/passwd HTTP/1.1" 400 312 "-" "python-requests/2.31.0"\n198.51.100.52 - - [04/Oct/2026:18:03:15 -0300] "GET /acesso HTTP/1.1" 200 7311 "https://serraazul.example/" "Mozilla/5.0 (Linux; Android 14; SM-A145M)"\n198.51.100.52 - - [04/Oct/2026:18:03:31 -0300] "POST /acesso HTTP/1.1" 302 0 "https://serraazul.example/acesso" "Mozilla/5.0 (Linux; Android 14; SM-A145M)"\n203.0.113.140 - - [04/Oct/2026:23:18:52 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (X11; Linux x86_64)"\n198.51.100.61 - - [05/Oct/2026:07:13:22 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (Linux; Android 14; SM-A145M)"\n198.51.100.17 - - [05/Oct/2026:07:14:05 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (iPhone; CPU iPhone OS 17_6 like Mac OS X)"\n198.51.100.88 - - [05/Oct/2026:07:19:47 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (Linux; Android 14; SM-A145M)"\n198.51.100.103 - - [05/Oct/2026:07:26:10 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (Linux; Android 14; SM-A145M)"\n198.51.100.17 - - [05/Oct/2026:07:31:58 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (iPhone; CPU iPhone OS 17_6 like Mac OS X)"\n198.51.100.144 - - [05/Oct/2026:07:38:30 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (iPhone; CPU iPhone OS 17_6 like Mac OS X)"\n198.51.100.72 - - [05/Oct/2026:07:44:02 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (Linux; Android 14; SM-A145M)"\n198.51.100.29 - - [05/Oct/2026:07:52:16 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (Linux; Android 14; SM-A145M)"\n198.51.100.199 - - [05/Oct/2026:08:03:41 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (iPhone; CPU iPhone OS 17_6 like Mac OS X)"\n198.51.100.66 - - [05/Oct/2026:08:11:09 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (Linux; Android 14; SM-A145M)"\n198.51.100.120 - - [05/Oct/2026:08:24:55 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (Linux; Android 14; SM-A145M)"\n198.51.100.35 - - [05/Oct/2026:08:39:12 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (iPhone; CPU iPhone OS 17_6 like Mac OS X)"\n198.51.100.81 - - [05/Oct/2026:08:47:30 -0300] "GET /static/img/logo-serraazul.png HTTP/1.1" 200 12877 "https://serraazul-pix.example/" "Mozilla/5.0 (Linux; Android 14; SM-A145M)"\n198.51.100.17 - - [05/Oct/2026:07:40:11 -0300] "GET / HTTP/1.1" 200 18422 "-" "Mozilla/5.0 (iPhone; CPU iPhone OS 17_6 like Mac OS X)"\n198.51.100.17 - - [05/Oct/2026:07:40:20 -0300] "GET /acesso HTTP/1.1" 200 7311 "https://serraazul.example/" "Mozilla/5.0 (iPhone; CPU iPhone OS 17_6 like Mac OS X)"\n198.51.100.17 - - [05/Oct/2026:07:40:51 -0300] "POST /acesso HTTP/1.1" 302 0 "https://serraazul.example/acesso" "Mozilla/5.0 (iPhone; CPU iPhone OS 17_6 like Mac OS X)"\n198.51.100.31 - - [05/Oct/2026:09:02:44 -0300] "GET / HTTP/1.1" 200 18422 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"\n198.51.100.31 - - [05/Oct/2026:09:02:45 -0300] "GET /avisos/golpe-pix HTTP/1.1" 404 512 "https://serraazul.example/" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"\n198.51.100.66 - - [05/Oct/2026:09:15:03 -0300] "GET / HTTP/1.1" 503 1024 "-" "Mozilla/5.0 (Linux; Android 14; SM-A145M)"\n198.51.100.66 - - [05/Oct/2026:09:15:09 -0300] "GET / HTTP/1.1" 200 18422 "-" "Mozilla/5.0 (Linux; Android 14; SM-A145M)"\n'

try:
    with urllib.request.urlopen(URL, timeout=8) as resp:   # timeout curto: sem rede, desiste logo
        LOG_SA = resp.read().decode("utf-8")
    origem = "GitHub"
except Exception as erro:                  # sem internet, repo privado, URL mudou...
    LOG_SA = EMBUTIDO                   # ...plano B: a cópia que veio dentro do notebook
    origem = f"cópia embutida ({type(erro).__name__})"

print(f"Carregado de: {origem} — {len(LOG_SA.splitlines())} linhas")

In [ ]:
import hashlib

sa = ler_log(LOG_SA)
hash_sa = hashlib.sha256(LOG_SA.encode("utf-8")).hexdigest()
print("SHA-256:", hash_sa)
print("Confere com o TI (c14588c5...f7ab65a3)?", hash_sa.startswith("c14588c5") and hash_sa.endswith("f7ab65a3"))
print("Intervalo UTC:", f"{sa[0]['utc']:%Y-%m-%dT%H:%M:%SZ}", "a", f"{sa[-1]['utc']:%Y-%m-%dT%H:%M:%SZ}")
print("Requisições:", len(sa), "| IPs únicos:", len({e['ip'] for e in sa}))

In [ ]:
# ------------------------------------------------------------
# DETECTOR 1 — espelhamento (alguém baixou o site para copiar)
# Forma: 1 IP, User-Agent de ferramenta, vários recursos em poucos segundos.
# ------------------------------------------------------------
FERRAMENTAS = ("wget", "curl", "httrack", "python-requests", "go-http-client")

por_ip = {}
for e in sa:
    if any(f in e["ua"].lower() for f in FERRAMENTAS):    # UA de ferramenta (pode ser forjado!)
        por_ip.setdefault(e["ip"], []).append(e)

for ip, evs in por_ip.items():
    estaticos = [e for e in evs if e["caminho"].startswith("/static/")]
    duracao = (evs[-1]["utc"] - evs[0]["utc"]).total_seconds()
    forma = "ESPELHAMENTO" if len(estaticos) >= 3 and duracao <= 10 else "varredura/outro"
    print(f"{ip:<15} {evs[0]['ua']:<24} {len(evs)} req em {duracao:.0f}s, {len(estaticos)} estáticos -> {forma}")
    if forma == "ESPELHAMENTO":
        for e in evs:
            print(f"      {e['utc']:%Y-%m-%dT%H:%M:%SZ}  {e['metodo']} {e['caminho']}")

In [ ]:
# ------------------------------------------------------------
# DETECTOR 2 — hotlink: o Referer é de um domínio que NÃO é o da marca
# Cada linha = um navegador que abriu a página falsa e pediu o logo ao site real.
# ------------------------------------------------------------
from urllib.parse import urlparse

DOMINIO_REAL = "serraazul.example"
hotlinks = []
for e in sa:
    host = urlparse(e["referer"]).hostname or ""          # "-" vira "" (sem Referer)
    if host and not (host == DOMINIO_REAL or host.endswith("." + DOMINIO_REAL)):
        hotlinks.append(e)
        print(f"{e['utc']:%Y-%m-%dT%H:%M:%SZ}  {e['ip']:<15} Referer={host}  UA={e['ua'][:40]}")

print(f"\n{len(hotlinks)} requisições, de {len({e['ip'] for e in hotlinks})} IPs distintos.")
print("Leia como: PELO MENOS esse número de navegadores abriu a página falsa (cache e bloqueadores somem do log).")

In [ ]:
# ------------------------------------------------------------
# DETECTOR 3 — quem viu a página falsa e depois foi ao site real?
# ------------------------------------------------------------
viu_golpe = {e["ip"]: e["utc"] for e in hotlinks}        # IP -> última vez que carregou a página falsa
for e in sa:
    if e["ip"] in viu_golpe and e["referer"] in ("-", "https://serraazul.example/", "https://serraazul.example/acesso") \
       and e["utc"] > viu_golpe[e["ip"]]:
        minutos = (e["utc"] - viu_golpe[e["ip"]]).total_seconds() / 60
        print(f"{e['ip']} foi ao portal real {minutos:.0f} min depois: {e['metodo']} {e['caminho']} -> {e['status']}")
print("\n🧠 Isso descreve um aparelho, não uma pessoa. O cruzamento com cadastro é do time de fraude da cooperativa.")

## 4. O histograma por hora (UTC)

In [ ]:
def histograma(eventos):
    """O mesmo do Passo 5, em Python: conta por (data, hora) e desenha barras de #."""
    contagem = Counter(f"{e['utc']:%Y-%m-%d %H}:00Z" for e in eventos)
    linhas = []
    for chave in sorted(contagem):                           # ordem cronológica
        linhas.append(f"{chave} {contagem[chave]:>3} {'#' * contagem[chave]}")
    return "\n".join(linhas)

HIST = histograma(sa)
print(HIST)
print("\nO pico de 10h UTC do dia 5 = 07h BRT: o disparo do SMS (07h12–08h40 BRT).")

## 5. 🎮 Quiz: qual comando responde?

Sete perguntas. Responda com a letra. `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("Ranquear IPs por número de requisições", ["a) awk '{print $1}' | sort | uniq -c | sort -rn", "b) grep -c '' access.log", "c) sed -n 1p access.log"], "a",
     "O idioma 'contar e ranquear'."),
    ("Ver o User-Agent de cada linha", ["a) awk '{print $6}'", "b) awk -F'\"' '{print $6}'", "c) awk '{print $12}'"], "b",
     "O UA tem espaços; separando por aspas ele vira o 6º campo inteiro."),
    ("Ver o Referer", ["a) awk -F'\"' '{print $4}'", "b) awk '{print $4}'", "c) awk -F'[][]' '{print $2}'"], "a",
     "Separando por aspas: $2 requisição, $4 Referer, $6 UA. awk '{print $4}' é a HORA."),
    ("Achar path traversal codificado", ["a) grep '../'", "b) grep -Ei '(\\.\\./|%2e%2e)'", "c) grep -v 200"], "b",
     "Precisa pegar a forma pura e a %2e%2e, sem diferenciar maiúsculas."),
    ("O log mostra 14 Referers de serraazul-pix.example. O relatório diz:", ["a) 14 vítimas", "b) pelo menos 13 navegadores abriram a página falsa", "c) 13 cooperados perderam dinheiro"], "b",
     "13 IPs distintos; é piso, e navegador não é vítima."),
    ("Há um CDN na frente do site. $1 é...", ["a) o cliente real", "b) o IP do CDN; o cliente só aparece se X-Forwarded-For foi registrado", "c) sempre o atacante"], "b",
     "Confira a arquitetura antes de concluir sobre origens."),
    ("O log BR está em -0300. 05/Oct/2026:07:13:22 -0300 em UTC é...", ["a) 2026-10-05T04:13:22Z", "b) 2026-10-05T10:13:22Z", "c) 2026-10-05T07:13:22Z"], "b",
     "UTC = hora local + 3 quando o fuso é -03:00."),
]
acertos = 0
for i, (p, ops, certa, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {p}")
    for o in ops:
        print("   ", o)
    if MODO_JOGO:
        r = ""
        while r not in ("a", "b", "c"):
            r = input("   Resposta (a/b/c): ").strip().lower()
        acertos += (r == certa)
        print("   ✅ Isso!" if r == certa else f"   ❌ Era {certa}.")
    print("   Por quê:", porque)
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}", "— Checkpoint BR cumprido." if acertos >= 5 else "— Meta: 5.")

## 6. 📝 A planilha de triagem do caso

In [ ]:
#@title Planilha de triagem — portal Serra Azul { display-mode: "form" }
achado_1 = "203.0.113.77 (IP de serraazul-pix.example) espelhou o portal com Wget/1.21.4 em 2 s, 2026-10-04T01:41:09Z a 01:41:11Z"  #@param {type:"string"}
achado_2 = "Pelo menos 13 navegadores (14 req.) abriram a página falsa: Referer serraazul-pix.example ao pedir o logo, 2026-10-05 10:13Z a 11:47Z"  #@param {type:"string"}
achado_3 = "203.0.113.140 carregou a página falsa em 2026-10-05T02:18:52Z, antes do disparo: provável teste do kit"  #@param {type:"string"}
achado_4 = "203.0.113.9: varredura genérica (wp-login, .env, traversal), só erros; sem relação demonstrada com o caso"  #@param {type:"string"}
linha_base = "198.51.100.31 e 198.51.100.52 navegam e fazem login como clientes comuns. O 503 de 198.51.100.66 às 12:15Z foi seguido de 200 seis segundos depois: instabilidade momentânea, não ataque."  #@param {type:"string"}

from pathlib import Path
planilha = f"""# Triagem de log web: portal serraazul.example (Dia 11)
SHA-256 do arquivo de origem: {hash_sa}
Intervalo coberto (UTC): {sa[0]['utc']:%Y-%m-%dT%H:%M:%SZ} a {sa[-1]['utc']:%Y-%m-%dT%H:%M:%SZ}
Total de requisições: {len(sa)} | IPs de cliente únicos: {len({e['ip'] for e in sa})}
Proxy/CDN na frente? Não (confirmado pelo TI); $1 é o cliente real.

## Achados ranqueados
| Posição | Achado | Confiança |
|---|---|---|
| 1 | {achado_1} | confirmado (7 linhas do log + IP do Dia 5) |
| 2 | {achado_2} | confirmado como piso |
| 3 | {achado_3} | provável |
| 4 | {achado_4} | confirmado (sem relação) |

## Atividade por hora (UTC)
```
{HIST}
```

## Linha de base benigna
{linha_base}
"""
Path("notas").mkdir(exist_ok=True)
Path("notas/isca-dia11-triagem.md").write_text(planilha, encoding="utf-8")
print(planilha)
try:
    from google.colab import files
    files.download("notas/isca-dia11-triagem.md")
except ImportError:
    print("(Fora do Colab: o arquivo ficou em notas/)")

### (Opcional) Os comandos de verdade, no shell do Colab

In [ ]:
!curl -sL https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-11/access-serraazul.log -o access-serraazul.log
!awk -F'"' '$4 ~ /serraazul-pix/ {split($1,a," "); print a[1]}' access-serraazul.log | sort | uniq -c | sort -rn
print("(Linhas com '!' só rodam no Colab/Linux.)")

## ✅ Antes de fechar o Dia 11

- [ ] Quiz com pelo menos 5/7
- [ ] `notas/isca-dia11-triagem.md` gerado, com o espelhamento em primeiro
- [ ] `day11-triage.md` da loja do original feito no shell (README, Passos 1 a 6)
- [ ] `status` do dia 11 = `estudado` no `curriculo/mapa-dos-dias.csv`

**Amanhã (Dia 12):** o `auth.log` do mesmo servidor. Alguém entrou como o Juliano às 22h47 de domingo. Foi ele?

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>